# 🤖 Лабораторная работа 8. Transformer и Attention

Цель: вручную пройти основные шаги Self-Attention и затем сравнить их с готовым `nn.MultiheadAttention`.


# 1. Импорт библиотек

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)

print("PyTorch:", torch.__version__)

# 2. Искусственные Embeddings

In [ ]:
batch_size = 1
sequence_length = 4
embedding_dim = 8

x = torch.randn(
    batch_size,
    sequence_length,
    embedding_dim,
)

print("Input shape:", x.shape)

# 3. Q, K и V

In [ ]:
query_layer = nn.Linear(embedding_dim, embedding_dim, bias=False)
key_layer = nn.Linear(embedding_dim, embedding_dim, bias=False)
value_layer = nn.Linear(embedding_dim, embedding_dim, bias=False)

Q = query_layer(x)
K = key_layer(x)
V = value_layer(x)

print("Q:", Q.shape)
print("K:", K.shape)
print("V:", V.shape)

# 4. Attention Scores

In [ ]:
scores = Q @ K.transpose(-2, -1)

print("Scores shape:", scores.shape)
print(scores)

# 5. Scale

In [ ]:
d_k = K.shape[-1]

scaled_scores = scores / math.sqrt(d_k)

print("d_k:", d_k)
print(scaled_scores)

# 6. Softmax

In [ ]:
attention_weights = torch.softmax(
    scaled_scores,
    dim=-1,
)

print("Weights shape:", attention_weights.shape)
print(attention_weights)

print(
    "Сумма первой строки:",
    attention_weights[0, 0].sum().item(),
)

# 7. Weighted Sum Values

In [ ]:
attention_output = attention_weights @ V

print("Output shape:", attention_output.shape)
print(attention_output)

# 8. Attention Matrix

In [ ]:
matrix = attention_weights[0].detach().numpy()

plt.figure(figsize=(7, 6))
plt.imshow(matrix)
plt.title("Attention Matrix")
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.colorbar()
plt.show()

# 9. Функция Self-Attention

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    d_k = K.shape[-1]

    scores = Q @ K.transpose(-2, -1)
    scores = scores / math.sqrt(d_k)

    weights = torch.softmax(
        scores,
        dim=-1,
    )

    output = weights @ V

    return output, weights


manual_output, manual_weights = scaled_dot_product_attention(
    Q,
    K,
    V,
)

print(manual_output.shape)
print(manual_weights.shape)

# 10. Causal Mask

In [ ]:
causal_mask = torch.triu(
    torch.ones(
        sequence_length,
        sequence_length,
    ),
    diagonal=1,
).bool()

print(causal_mask)

# 11. Attention с Causal Mask

In [ ]:
masked_scores = scaled_scores.masked_fill(
    causal_mask,
    float("-inf"),
)

causal_weights = torch.softmax(
    masked_scores,
    dim=-1,
)

print(causal_weights)

# 12. Визуализация Causal Attention

In [ ]:
matrix = causal_weights[0].detach().numpy()

plt.figure(figsize=(7, 6))
plt.imshow(matrix)
plt.title("Causal Attention Matrix")
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.colorbar()
plt.show()

# 13. nn.MultiheadAttention

In [ ]:
multihead = nn.MultiheadAttention(
    embed_dim=embedding_dim,
    num_heads=2,
    batch_first=True,
)

mha_output, mha_weights = multihead(
    x,
    x,
    x,
)

print("Input:", x.shape)
print("Output:", mha_output.shape)
print("Weights:", mha_weights.shape)

# 14. Простейший Transformer Block

In [ ]:
class TinyTransformerBlock(nn.Module):
    def __init__(
        self,
        embedding_dim=8,
        num_heads=2,
        ff_dim=16,
    ):
        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            batch_first=True,
        )

        self.norm1 = nn.LayerNorm(embedding_dim)

        self.feed_forward = nn.Sequential(
            nn.Linear(embedding_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embedding_dim),
        )

        self.norm2 = nn.LayerNorm(embedding_dim)

    def forward(self, x):
        attention_output, weights = self.attention(
            x,
            x,
            x,
        )

        x = self.norm1(
            x + attention_output
        )

        ff_output = self.feed_forward(x)

        x = self.norm2(
            x + ff_output
        )

        return x, weights


block = TinyTransformerBlock()

block_output, block_weights = block(x)

print("Input:", x.shape)
print("Output:", block_output.shape)
print("Attention weights:", block_weights.shape)

# 15. Количество параметров

In [ ]:
total_parameters = sum(
    p.numel()
    for p in block.parameters()
)

print("Всего параметров:", total_parameters)

# 16. 📌 Что нужно запомнить

```text
Embeddings
↓
Q, K, V
↓
Scores = QKᵀ
↓
Scale
↓
Softmax
↓
Attention Weights
↓
Weights × V
↓
Contextual Representations
```


# 17. 🧩 Эксперименты

Попробуй:

- изменить `sequence_length`;
- изменить `embedding_dim`;
- изменить число Heads;
- сравнить обычный и Causal Attention;
- посмотреть, как меняется Attention Matrix;
- заменить ReLU на GELU в Feed-Forward.


# 18. ➡️ Следующая глава

# Глава 9. Локальные LLM и Qwen
